# Start-up credit risk: walkthrough

This notebook runs the pipeline step by step and shows the main results. Put the raw data at `../data/startup_failures.csv` first (see `data/README.md`).

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from startup_credit_risk import config, model, scoring, products
from startup_credit_risk.preprocessing import build_dataset

DATA = '../data/startup_failures.csv'

## 1. Data after cleaning and scoping

In [ ]:
df = build_dataset(DATA)
print(f'{len(df):,} start-ups, failure rate {df[config.TARGET].mean():.1%}')
df.describe().T

## 2. Train and calibrate the model

In [ ]:
splits = model.split_data(df)
base, calibrated = model.train(splits)

pd.DataFrame({
    'uncalibrated': model.evaluate(base, splits.X_test, splits.y_test),
    'calibrated': model.evaluate(calibrated, splits.X_test, splits.y_test),
}).round(4)

In [ ]:
model.feature_importance(base).head(10)

## 3. Risk buckets (test set)

A well-built rating scale shows the observed failure rate rising steadily from A to F, close to the predicted rate.

In [ ]:
p_test = calibrated.predict_proba(splits.X_test)[:, 1]
scoring.bucket_summary(p_test, splits.y_test)

## 4. Loan portfolio

In [ ]:
scored = df.copy()
scored['failure_probability'] = calibrated.predict_proba(scored[splits.X_test.columns])[:, 1]
scored['bucket'] = scoring.assign_bucket(scoring.credit_score(scored['failure_probability'])).to_numpy()
products.portfolio(scored)